# RHI Live Runtime v22 — Passive Intermediate Fold Logger

Δ **Purpose:** instrument token-level generation to observe fold structure, not just terminal collapse.

$$
\boxed{
\text{The answer is not the fold. The answer is the terminal shadow of the fold.}
}
$$

## What v22 Does

v21 audits only at terminal:

$$
G(\Psi_{\text{final}})
$$

v22 logs the fold path:

$$
\{R_\ell, S_\ell, C_\ell, U_\ell\}_{\ell=1}^{L}
$$

where:

- $R_\ell$ = token at position $\ell$
- $S_\ell$ = logit entropy at position $\ell$
- $C_\ell$ = top-token confidence at position $\ell$
- $U_\ell$ = drift indicators (carrier terms, scorer terms)

## Core Measurements

### 1. Token Entropy Profile

$$
S_\ell = -\sum_{i} p_i^{(\ell)} \log p_i^{(\ell)}
$$

**Hypothesis:** Successful $\Psi$ collapses show entropy convergence at characteristic levels.

### 2. Confidence Trajectory

$$
C_\ell = \max_i p_i^{(\ell)}
$$

**Hypothesis:** Checklist-like answers show high confidence on scorer terms.

### 3. Semantic Carrier Drift

Track when scorer terms appear: `precondition`, `rollback`, `criteria`, `boundary`, `contract`.

$$
\text{drift}_\ell = \mathbb{1}[\text{token}_\ell \in \text{scorer vocabulary}]
$$

### 4. H-Ratio Search

For each branch, compute:

$$
H_\ell^{(\text{entropy})} = \frac{S_\ell}{S_{\max}}
$$

$$
H_\ell^{(\text{confidence})} = C_\ell
$$

Find:

$$
\ell^* = \arg\min_\ell |H_\ell - \pi/9|
$$

**Conjecture:** Successful collapses show $H$-convergence at intermediate levels.

## Output

Still exactly two files:

1. `rhi_v22_<run_id>_bundle.json` — includes per-token fold logs
2. `rhi_v22_<run_id>_summary.csv` — includes fold metrics

## The Lock

$$
\boxed{
\text{This is passive instrumentation. No intervention. No abort. Just observe.}
}
$$

In [ ]:
from __future__ import annotations
import os, re, sys, json, math, uuid, time, random, traceback, subprocess, importlib
from dataclasses import dataclass, asdict, field
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple
import numpy as np

ROOT = Path.cwd()
OUT_DIR = ROOT / 'rhi_v22_outputs'
OUT_DIR.mkdir(exist_ok=True)
RUN_ID = 'rhi_v22_' + uuid.uuid4().hex[:10]
SEED = 7
random.seed(SEED)
np.random.seed(SEED)

MODEL_ID_OR_PATH = os.environ.get('RHI_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')
LOAD_REAL_MODEL = True
REQUIRE_MODEL_FOR_PSI = True
AUTO_INSTALL_MISSING_DEPS = True
RUN_PROMPT_BATTERY = True

# Fold instrumentation constants
H_TARGET = math.pi / 9  # ≈ 0.34906585
SCORER_TERMS = {
    'precondition', 'rollback', 'criteria', 'boundary', 'contract',
    'validation', 'constraint', 'invariant', 'audit', 'verify'
}

print(f"RHI v22 Passive Fold Logger")
print(f"Run ID: {RUN_ID}")
print(f"H target: {H_TARGET:.8f}")
print(f"Output dir: {OUT_DIR}")

In [ ]:
# Token-level fold state
@dataclass
class TokenFoldState:
    """State at token position ℓ during generation."""
    position: int  # ℓ
    token_id: int
    token_text: str
    logit_entropy: float  # S_ℓ
    top_confidence: float  # C_ℓ = max p_i
    is_scorer_term: bool  # drift indicator
    
    # H-ratios
    h_entropy_ratio: Optional[float] = None  # S_ℓ / S_max
    h_confidence_ratio: Optional[float] = None  # C_ℓ
    h_distance: Optional[float] = None  # min |H_ℓ - π/9|

@dataclass
class BranchFoldLog:
    """Complete fold trajectory for one branch."""
    branch_role: str
    branch_id: str
    fold_states: List[TokenFoldState] = field(default_factory=list)
    
    # Aggregate metrics
    total_tokens: int = 0
    scorer_term_count: int = 0
    mean_entropy: float = 0.0
    mean_confidence: float = 0.0
    
    # H-convergence
    min_h_distance: float = float('inf')
    h_convergence_position: Optional[int] = None
    h_convergence_metric: Optional[str] = None
    
    # Terminal state
    final_text: str = ""
    final_state: str = ""  # Ψ or Ω
    is_checklist_like: bool = False

In [ ]:
def compute_entropy(logits: np.ndarray, temperature: float = 1.0) -> float:
    """Compute Shannon entropy of softmax distribution.
    
    S = -∑ p_i log p_i
    """
    logits = logits / temperature
    # Subtract max for numerical stability
    logits_shifted = logits - np.max(logits)
    exp_logits = np.exp(logits_shifted)
    probs = exp_logits / np.sum(exp_logits)
    
    # Avoid log(0)
    probs = np.clip(probs, 1e-10, 1.0)
    entropy = -np.sum(probs * np.log(probs))
    return float(entropy)

def get_top_confidence(logits: np.ndarray, temperature: float = 1.0) -> float:
    """Get confidence of most likely token.
    
    C = max_i p_i
    """
    logits = logits / temperature
    logits_shifted = logits - np.max(logits)
    exp_logits = np.exp(logits_shifted)
    probs = exp_logits / np.sum(exp_logits)
    return float(np.max(probs))

def is_scorer_term(token_text: str, scorer_vocab: set) -> bool:
    """Check if token is a scorer vocabulary term."""
    token_lower = token_text.lower().strip()
    return token_lower in scorer_vocab or any(
        term in token_lower for term in scorer_vocab
    )

def compute_h_distance(value: float, target: float = H_TARGET) -> float:
    """Distance from H = π/9."""
    return abs(value - target)

In [ ]:
class FoldInstrumentedGenerator:
    """Wrapper around HuggingFace generate() that logs fold states."""
    
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.fold_log = BranchFoldLog(branch_role="", branch_id="")
        
    def generate_with_fold_logging(
        self,
        prompt: str,
        branch_role: str,
        branch_id: str,
        max_new_tokens: int = 200,
        temperature: float = 0.7,
    ) -> Tuple[str, BranchFoldLog]:
        """Generate text while logging token-level fold states."""
        
        self.fold_log = BranchFoldLog(
            branch_role=branch_role,
            branch_id=branch_id
        )
        
        # Tokenize prompt
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        
        # Generate with logit tracking
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            return_dict_in_generate=True,
            output_scores=True,
        )
        
        # Extract generated tokens (exclude prompt)
        generated_ids = outputs.sequences[0][inputs.input_ids.shape[1]:]
        scores = outputs.scores  # Tuple of (batch_size, vocab_size) tensors
        
        # Log each token
        for position, (token_id, logits_tensor) in enumerate(zip(generated_ids, scores), start=1):
            # Convert to numpy
            logits = logits_tensor[0].cpu().numpy()  # Shape: (vocab_size,)
            
            # Compute metrics
            entropy = compute_entropy(logits, temperature)
            confidence = get_top_confidence(logits, temperature)
            
            # Decode token
            token_text = self.tokenizer.decode([token_id.item()])
            
            # Check scorer drift
            is_scorer = is_scorer_term(token_text, SCORER_TERMS)
            
            # Create fold state
            fold_state = TokenFoldState(
                position=position,
                token_id=token_id.item(),
                token_text=token_text,
                logit_entropy=entropy,
                top_confidence=confidence,
                is_scorer_term=is_scorer,
            )
            
            self.fold_log.fold_states.append(fold_state)
        
        # Compute aggregate metrics
        self._compute_aggregate_metrics()
        
        # Decode full text
        generated_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        self.fold_log.final_text = generated_text
        
        return generated_text, self.fold_log
    
    def _compute_aggregate_metrics(self):
        """Compute summary statistics over fold trajectory."""
        if not self.fold_log.fold_states:
            return
        
        states = self.fold_log.fold_states
        
        # Basic counts
        self.fold_log.total_tokens = len(states)
        self.fold_log.scorer_term_count = sum(s.is_scorer_term for s in states)
        
        # Entropy/confidence means
        entropies = [s.logit_entropy for s in states]
        confidences = [s.top_confidence for s in states]
        
        self.fold_log.mean_entropy = np.mean(entropies)
        self.fold_log.mean_confidence = np.mean(confidences)
        
        # Compute H-ratios for each position
        max_entropy = max(entropies) if entropies else 1.0
        
        min_h_dist = float('inf')
        h_conv_pos = None
        h_conv_metric = None
        
        for state in states:
            # Normalize entropy
            h_entropy = state.logit_entropy / max_entropy if max_entropy > 0 else 0
            state.h_entropy_ratio = h_entropy
            
            # Confidence is already [0,1]
            h_confidence = state.top_confidence
            state.h_confidence_ratio = h_confidence
            
            # Find minimum distance to H
            h_dist_entropy = compute_h_distance(h_entropy)
            h_dist_confidence = compute_h_distance(h_confidence)
            
            state.h_distance = min(h_dist_entropy, h_dist_confidence)
            
            if state.h_distance < min_h_dist:
                min_h_dist = state.h_distance
                h_conv_pos = state.position
                h_conv_metric = 'entropy' if h_dist_entropy < h_dist_confidence else 'confidence'
        
        self.fold_log.min_h_distance = min_h_dist
        self.fold_log.h_convergence_position = h_conv_pos
        self.fold_log.h_convergence_metric = h_conv_metric
        
        # Detect checklist pattern
        # Heuristic: >30% scorer terms or >5 consecutive scorer terms
        scorer_ratio = self.fold_log.scorer_term_count / self.fold_log.total_tokens
        
        consecutive_scorer = 0
        max_consecutive = 0
        for s in states:
            if s.is_scorer_term:
                consecutive_scorer += 1
                max_consecutive = max(max_consecutive, consecutive_scorer)
            else:
                consecutive_scorer = 0
        
        self.fold_log.is_checklist_like = (scorer_ratio > 0.3) or (max_consecutive > 5)

In [ ]:
# Install dependencies if needed
def ensure_deps():
    """Install required packages."""
    if not AUTO_INSTALL_MISSING_DEPS:
        return
    
    try:
        import torch
        import transformers
    except ImportError:
        print("Installing torch and transformers...")
        subprocess.run([
            sys.executable, "-m", "pip", "install", 
            "torch", "transformers", "--break-system-packages"
        ], check=True)
        # Reload
        import torch
        import transformers

ensure_deps()

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load model
def load_model():
    """Load model and tokenizer."""
    if not LOAD_REAL_MODEL:
        return None, None
    
    print(f"Loading model: {MODEL_ID_OR_PATH}")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_OR_PATH)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID_OR_PATH,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    
    if device == "cpu":
        model = model.to(device)
    
    model.eval()
    
    print(f"Model loaded on {device}")
    return model, tokenizer

model, tokenizer = load_model()

In [ ]:
# Test prompts from v21
TEST_PROMPTS = [
    {
        "id": "runtime_contract_1",
        "profile": "runtime_contract",
        "prompt": "explain how a runtime contract differs from a legal contract in the context of LLM inference control"
    },
    {
        "id": "tool_safety_1",
        "profile": "tool_safety",
        "prompt": "describe the safety requirements for a tool that can execute arbitrary shell commands"
    },
    {
        "id": "evidence_control_1",
        "profile": "evidence_control",
        "prompt": "explain how to verify that an AI agent's answer is based on retrieved evidence and not hallucination"
    },
    {
        "id": "retrieval_inverse_1",
        "profile": "retrieval",
        "prompt": "design a shape-first retrieval step where no noun match exists but the inverse need is clear"
    },
    {
        "id": "state_recovery_1",
        "profile": "state_recovery",
        "prompt": "describe how to recover system state after a failed database transaction without losing user context"
    },
]

In [ ]:
# Branch roles (simplified for v22 - focus on logging, not complex branching)
BRANCH_ROLES = ['construct', 'verify']  # Just two branches to keep it manageable

def generate_branch_prompt(base_prompt: str, role: str) -> str:
    """Adapt prompt for branch role."""
    if role == 'construct':
        return f"{base_prompt}\n\nProvide a clear, direct answer."
    elif role == 'verify':
        return f"{base_prompt}\n\nFirst verify the requirements, then provide your answer."
    return base_prompt

In [ ]:
# Run fold-instrumented generation
all_fold_logs = []

if LOAD_REAL_MODEL and model is not None:
    generator = FoldInstrumentedGenerator(model, tokenizer)
    
    for test_case in TEST_PROMPTS:
        print(f"\n{'='*60}")
        print(f"Prompt: {test_case['id']}")
        print(f"Profile: {test_case['profile']}")
        print(f"{'='*60}")
        
        for role in BRANCH_ROLES:
            branch_id = f"{test_case['id']}_{role}"
            branch_prompt = generate_branch_prompt(test_case['prompt'], role)
            
            print(f"\nBranch: {role}")
            print("-" * 40)
            
            try:
                generated_text, fold_log = generator.generate_with_fold_logging(
                    prompt=branch_prompt,
                    branch_role=role,
                    branch_id=branch_id,
                    max_new_tokens=150,
                    temperature=0.7,
                )
                
                # Print summary
                print(f"Generated {fold_log.total_tokens} tokens")
                print(f"Scorer terms: {fold_log.scorer_term_count} ({fold_log.scorer_term_count/fold_log.total_tokens*100:.1f}%)")
                print(f"Mean entropy: {fold_log.mean_entropy:.4f}")
                print(f"Mean confidence: {fold_log.mean_confidence:.4f}")
                print(f"Checklist-like: {fold_log.is_checklist_like}")
                print(f"H-convergence: position {fold_log.h_convergence_position}, metric {fold_log.h_convergence_metric}, distance {fold_log.min_h_distance:.6f}")
                print(f"\nGenerated text preview:")
                print(generated_text[:200] + "..." if len(generated_text) > 200 else generated_text)
                
                # Store log
                all_fold_logs.append(fold_log)
                
            except Exception as e:
                print(f"Error generating branch: {e}")
                traceback.print_exc()

print(f"\n{'='*60}")
print(f"Completed {len(all_fold_logs)} branch generations")
print(f"{'='*60}")

In [ ]:
# Analyze H-convergence patterns
print("\n" + "="*60)
print("H-CONVERGENCE ANALYSIS")
print("="*60)

h_distances = [log.min_h_distance for log in all_fold_logs if log.min_h_distance < float('inf')]
h_positions = [log.h_convergence_position for log in all_fold_logs if log.h_convergence_position is not None]
h_metrics = [log.h_convergence_metric for log in all_fold_logs if log.h_convergence_metric is not None]

if h_distances:
    print(f"\nMinimum H-distance statistics:")
    print(f"  Mean: {np.mean(h_distances):.6f}")
    print(f"  Median: {np.median(h_distances):.6f}")
    print(f"  Min: {np.min(h_distances):.6f}")
    print(f"  Max: {np.max(h_distances):.6f}")
    
    # Count how many come within threshold of H
    close_threshold = 0.05  # Within 5% of H
    close_count = sum(1 for d in h_distances if d < close_threshold)
    print(f"\nBranches within {close_threshold} of H: {close_count}/{len(h_distances)} ({close_count/len(h_distances)*100:.1f}%)")

if h_positions:
    print(f"\nH-convergence position statistics:")
    print(f"  Mean position: {np.mean(h_positions):.1f}")
    print(f"  Median position: {np.median(h_positions):.1f}")
    
    # As fraction of total length
    total_lengths = [log.total_tokens for log in all_fold_logs if log.h_convergence_position is not None]
    if total_lengths:
        relative_positions = [pos/total for pos, total in zip(h_positions, total_lengths)]
        print(f"  Mean relative position: {np.mean(relative_positions):.3f}")
        print(f"  (H target = {H_TARGET:.3f} for comparison)")

if h_metrics:
    metric_counts = Counter(h_metrics)
    print(f"\nH-convergence by metric:")
    for metric, count in metric_counts.items():
        print(f"  {metric}: {count} ({count/len(h_metrics)*100:.1f}%)")

In [ ]:
# Checklist analysis
print("\n" + "="*60)
print("CHECKLIST DRIFT ANALYSIS")
print("="*60)

checklist_count = sum(1 for log in all_fold_logs if log.is_checklist_like)
print(f"\nBranches detected as checklist-like: {checklist_count}/{len(all_fold_logs)} ({checklist_count/len(all_fold_logs)*100:.1f}%)")

# Group by profile
from collections import defaultdict
profile_checklists = defaultdict(list)
for i, log in enumerate(all_fold_logs):
    # Extract profile from branch_id
    parts = log.branch_id.split('_')
    if len(parts) >= 2:
        profile = '_'.join(parts[:-1])  # Everything except last part (role)
        profile_checklists[profile].append(log.is_checklist_like)

print("\nChecklist tendency by prompt profile:")
for profile, values in sorted(profile_checklists.items()):
    checklist_ratio = sum(values) / len(values) if values else 0
    print(f"  {profile}: {sum(values)}/{len(values)} ({checklist_ratio*100:.1f}%)")

# Scorer term statistics
scorer_ratios = [log.scorer_term_count / log.total_tokens for log in all_fold_logs if log.total_tokens > 0]
if scorer_ratios:
    print(f"\nScorer term density:")
    print(f"  Mean: {np.mean(scorer_ratios)*100:.1f}%")
    print(f"  Median: {np.median(scorer_ratios)*100:.1f}%")
    print(f"  Range: {np.min(scorer_ratios)*100:.1f}% - {np.max(scorer_ratios)*100:.1f}%")

In [ ]:
# Write bundle.json
bundle_path = OUT_DIR / f"{RUN_ID}_bundle.json"

bundle = {
    "run_id": RUN_ID,
    "version": "v22",
    "purpose": "passive_intermediate_fold_logging",
    "model": MODEL_ID_OR_PATH,
    "h_target": H_TARGET,
    "scorer_terms": list(SCORER_TERMS),
    "fold_logs": [
        {
            "branch_id": log.branch_id,
            "branch_role": log.branch_role,
            "total_tokens": log.total_tokens,
            "scorer_term_count": log.scorer_term_count,
            "mean_entropy": log.mean_entropy,
            "mean_confidence": log.mean_confidence,
            "is_checklist_like": log.is_checklist_like,
            "min_h_distance": log.min_h_distance,
            "h_convergence_position": log.h_convergence_position,
            "h_convergence_metric": log.h_convergence_metric,
            "final_text": log.final_text,
            "fold_states": [
                {
                    "position": s.position,
                    "token_text": s.token_text,
                    "logit_entropy": s.logit_entropy,
                    "top_confidence": s.top_confidence,
                    "is_scorer_term": s.is_scorer_term,
                    "h_entropy_ratio": s.h_entropy_ratio,
                    "h_confidence_ratio": s.h_confidence_ratio,
                    "h_distance": s.h_distance,
                }
                for s in log.fold_states
            ]
        }
        for log in all_fold_logs
    ]
}

with open(bundle_path, 'w') as f:
    json.dump(bundle, f, indent=2)

print(f"\nBundle written: {bundle_path}")
print(f"Size: {bundle_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Write summary.csv
import csv

summary_path = OUT_DIR / f"{RUN_ID}_summary.csv"

with open(summary_path, 'w', newline='') as f:
    writer = csv.writer(f)
    
    # Header
    writer.writerow([
        'branch_id',
        'branch_role',
        'total_tokens',
        'scorer_term_count',
        'scorer_term_ratio',
        'mean_entropy',
        'mean_confidence',
        'is_checklist_like',
        'min_h_distance',
        'h_convergence_position',
        'h_convergence_relative',
        'h_convergence_metric',
    ])
    
    # Data
    for log in all_fold_logs:
        scorer_ratio = log.scorer_term_count / log.total_tokens if log.total_tokens > 0 else 0
        h_conv_rel = log.h_convergence_position / log.total_tokens if log.h_convergence_position and log.total_tokens > 0 else None
        
        writer.writerow([
            log.branch_id,
            log.branch_role,
            log.total_tokens,
            log.scorer_term_count,
            f"{scorer_ratio:.4f}",
            f"{log.mean_entropy:.4f}",
            f"{log.mean_confidence:.4f}",
            log.is_checklist_like,
            f"{log.min_h_distance:.6f}" if log.min_h_distance < float('inf') else '',
            log.h_convergence_position or '',
            f"{h_conv_rel:.4f}" if h_conv_rel is not None else '',
            log.h_convergence_metric or '',
        ])

print(f"Summary written: {summary_path}")
print(f"Size: {summary_path.stat().st_size / 1024:.1f} KB")

## v22 Complete

This run logged token-level fold states for all branches:

$$
\{R_\ell, S_\ell, C_\ell, U_\ell\}_{\ell=1}^{L}
$$

Key findings to check:

1. **H-convergence:** Do successful branches show $H_\ell \approx \pi/9$ at intermediate positions?
2. **Checklist drift:** Can we detect scorer-term accumulation before terminal?
3. **Entropy profiles:** Do they show convergence patterns?
4. **Relative position:** Is $\ell^*/L \approx \pi/9$?

The bundle.json contains full token-by-token logs for deeper analysis.

Next: v23 should add:
- Attention entropy (if extractable from HuggingFace)
- Cross-branch divergence tracking
- Earlier abort intervention based on drift thresholds